# Experiment 3: block-diagonal-covariance AR(1)

Tests the RQ1 null under a second covariance family, to replace the claim that
compound symmetry is "most hostile to CD" with evidence. Channels split into
groups of seven; correlation is rho_in within a group and zero between groups.
The transition stays diagonal (phi I), so Granger non-causality holds exactly and
any CD advantage would have to come from instantaneous correlation alone.

**Cells:** C in {21, 84}, rho_in in {0.5, 0.9}, group_size=7, rho_out=0.
**Modes:** CI, CD, DLinear. **Seeds:** {42,123,456,789,1011}. 60 runs.
**Output:** /kaggle/working/results_block_cov.csv


In [ ]:
# Imports and device setup
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def free_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
# Sequence and architecture (Table 1, synthetic + ETTh1 config) -- identical to the canonical grid.
SEQ_LEN:      int = 512
PRED_LEN:     int = 96
PATCH_SIZE:   int = 16
PATCH_STRIDE: int = 8

D_MODEL:  int   = 64
N_HEADS:  int   = 8
N_LAYERS: int   = 3
DROPOUT:  float = 0.2

LR:            float = 1e-4
WEIGHT_DECAY:  float = 1e-4
MAX_EPOCHS:    int   = 50
PATIENCE:      int   = 10
GRAD_CLIP:     float = 1.0
WARMUP_EPOCHS: int   = 10   # CI/CD only; DLinear uses no warmup

N_PATCHES = (SEQ_LEN - PATCH_SIZE) // PATCH_STRIDE + 1   # 63, independent of C


In [ ]:
# Data-generating-process constants (identical to the canonical grid).
PHI:        float = 0.8
T_TOTAL:    int   = 14_400
BURN_IN:    int   = 1_000
TRAIN_FRAC: float = 0.6
VAL_FRAC:   float = 0.2

SEEDS: list[int] = [42, 123, 456, 789, 1011]
MODES: list[str] = ["CI", "CD", "DLinear"]
GROUP_SIZE: int = 7
RHO_OUT: float = 0.0

BATCH_BY_C: dict[int, dict[str, int]] = {
    21: {"CI": 128, "CD": 8, "DLinear": 128},
    84: {"CI": 32,  "CD": 1, "DLinear": 128},
}

CELLS = [{"cell": f"block_C{C}_rhoin{r}", "C": C, "rho_in": r} for C in (21, 84) for r in (0.5, 0.9)]
print(f"runs = {len(CELLS) * len(MODES) * len(SEEDS)}  cells={[c['cell'] for c in CELLS]}")


In [ ]:
def _ar1_transition(phi: float, C: int) -> np.ndarray:
    return np.eye(C, dtype=np.float64) * phi


def _block_cov(C: int, group_size: int, rho_in: float, rho_out: float) -> np.ndarray:
    if C % group_size != 0:
        raise ValueError(f"C={C} not divisible by group_size={group_size}")
    if not (rho_out < rho_in):
        raise ValueError("require rho_out < rho_in")
    gid = np.repeat(np.arange(C // group_size), group_size)
    same = gid[:, None] == gid[None, :]
    sigma = np.where(same, rho_in, rho_out).astype(np.float64)
    np.fill_diagonal(sigma, 1.0)
    if np.linalg.eigvalsh(sigma).min() <= 0:
        raise ValueError(f"Block covariance not PD: rho_in={rho_in}, rho_out={rho_out}, group_size={group_size}")
    return sigma


def generate_block(C: int, rho_in: float, rho_out: float, group_size: int, seed: int) -> np.ndarray:
    """Diagonal-transition AR(1) with block-diagonal innovations. (T_TOTAL, C)."""
    rng = np.random.default_rng(seed)
    A = _ar1_transition(PHI, C)
    L = np.linalg.cholesky(_block_cov(C, group_size, rho_in, rho_out))
    X = np.zeros((T_TOTAL + BURN_IN, C), dtype=np.float64)
    for t in range(1, T_TOTAL + BURN_IN):
        X[t] = A @ X[t - 1] + L @ rng.standard_normal(C)
    return X[BURN_IN:]


def split_normalise(data: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    n_train = int(len(data) * TRAIN_FRAC)
    n_val   = int(len(data) * VAL_FRAC)
    train, val, test = data[:n_train], data[n_train:n_train + n_val], data[n_train + n_val:]
    mean = train.mean(axis=0, keepdims=True)
    std  = np.where(train.std(axis=0, keepdims=True) == 0, 1.0, train.std(axis=0, keepdims=True))
    return (train - mean) / std, (val - mean) / std, (test - mean) / std


def make_windows(data: np.ndarray) -> tuple[torch.Tensor, torch.Tensor]:
    """Stride-tricks windowing (no Python loop)."""
    T, Cv = data.shape
    n = T - SEQ_LEN - PRED_LEN + 1
    if n <= 0:
        raise ValueError(f"Not enough timesteps: {T}")
    s0, s1 = data.strides
    view = np.lib.stride_tricks.as_strided(data, shape=(n, SEQ_LEN + PRED_LEN, Cv), strides=(s0, s0, s1))
    xs = np.ascontiguousarray(view[:, :SEQ_LEN])
    ys = np.ascontiguousarray(view[:, SEQ_LEN:])
    return torch.tensor(xs, dtype=torch.float32), torch.tensor(ys, dtype=torch.float32)


In [ ]:
# Model definitions. CI/CD/DLinear are byte-faithful to train_leader_follower.
# CD_Head adds an explicit cross-variate pool at the head; the encoder is unchanged.

class PatchEmbedding(nn.Module):
    def __init__(self, patch_size: int, d_model: int, dropout: float) -> None:
        super().__init__()
        self.proj    = nn.Linear(patch_size, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.proj(x))


def _encoder() -> nn.TransformerEncoder:
    layer = nn.TransformerEncoderLayer(
        d_model=D_MODEL, nhead=N_HEADS, dim_feedforward=D_MODEL * 4,
        dropout=DROPOUT, batch_first=True)
    return nn.TransformerEncoder(layer, num_layers=N_LAYERS)


class PatchTST_CI(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.embed   = PatchEmbedding(PATCH_SIZE, D_MODEL, DROPOUT)
        self.encoder = _encoder()
        self.head    = nn.Linear(N_PATCHES * D_MODEL, PRED_LEN)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, Cv = x.shape
        f = x.permute(0, 2, 1).reshape(B * Cv, L)
        p = f.unfold(-1, PATCH_SIZE, PATCH_STRIDE)
        e = self.encoder(self.embed(p))
        return self.head(e.reshape(B * Cv, -1)).reshape(B, Cv, -1).permute(0, 2, 1)


class PatchTST_CD(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.embed   = PatchEmbedding(PATCH_SIZE, D_MODEL, DROPOUT)
        self.encoder = _encoder()
        self.head    = nn.Linear(N_PATCHES * D_MODEL, PRED_LEN)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, Cv = x.shape
        f   = x.permute(0, 2, 1).reshape(B * Cv, L)
        p   = f.unfold(-1, PATCH_SIZE, PATCH_STRIDE)
        emb = self.embed(p).reshape(B, Cv, N_PATCHES, -1)
        seq = emb.reshape(B, Cv * N_PATCHES, -1)
        enc = self.encoder(seq).reshape(B * Cv, -1)
        return self.head(enc).reshape(B, Cv, -1).permute(0, 2, 1)


class PatchTST_CD_Head(nn.Module):
    """CD encoder unchanged; explicit cross-variate pooling added at the head only."""

    def __init__(self) -> None:
        super().__init__()
        self.embed        = PatchEmbedding(PATCH_SIZE, D_MODEL, DROPOUT)
        self.encoder      = _encoder()
        self.summary_norm = nn.LayerNorm(D_MODEL)
        self.channel_attn = nn.MultiheadAttention(D_MODEL, N_HEADS, dropout=DROPOUT, batch_first=True)
        self.head         = nn.Linear(N_PATCHES * D_MODEL + D_MODEL, PRED_LEN)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, Cv = x.shape
        f   = x.permute(0, 2, 1).reshape(B * Cv, L)
        p   = f.unfold(-1, PATCH_SIZE, PATCH_STRIDE)
        emb = self.embed(p).reshape(B, Cv, N_PATCHES, -1)
        seq = emb.reshape(B, Cv * N_PATCHES, -1)
        enc = self.encoder(seq).reshape(B, Cv, N_PATCHES, D_MODEL)          # (B, C, N, D)
        summary  = self.summary_norm(enc.mean(dim=2))                        # (B, C, D)
        ctx, _   = self.channel_attn(summary, summary, summary)              # (B, C, D)
        flat     = enc.reshape(B, Cv, N_PATCHES * D_MODEL)                   # (B, C, N*D)
        fused    = torch.cat([flat, ctx], dim=-1)                           # (B, C, N*D + D)
        return self.head(fused.reshape(B * Cv, -1)).reshape(B, Cv, -1).permute(0, 2, 1)


class TrueDLinear(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        pad = (25 - 1) // 2
        self.avg_pool         = nn.AvgPool1d(kernel_size=25, stride=1, padding=pad)
        self.linear_trend     = nn.Linear(SEQ_LEN, PRED_LEN)
        self.linear_remainder = nn.Linear(SEQ_LEN, PRED_LEN)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, Cv = x.shape
        xf    = x.permute(0, 2, 1).reshape(B * Cv, 1, L)
        trend = self.avg_pool(xf).reshape(B * Cv, L)[:, :L]
        rem   = xf.reshape(B * Cv, L) - trend
        out   = self.linear_trend(trend) + self.linear_remainder(rem)
        return out.reshape(B, Cv, -1).permute(0, 2, 1)


def build_model(mode: str) -> nn.Module:
    if mode == "CI":      return PatchTST_CI()
    if mode == "CD":      return PatchTST_CD()
    if mode == "CD_Head": return PatchTST_CD_Head()
    if mode == "DLinear": return TrueDLinear()
    raise ValueError(f"Unknown mode: {mode}")


In [ ]:
# Architecture assertions: fail fast before any training.
_x = torch.zeros(2, SEQ_LEN, 21)
for _m in MODES:
    _net = build_model(_m)
    assert _net(_x).shape == (2, PRED_LEN, 21), f"{_m} output shape wrong"
    del _net
_cd = build_model("CD")
assert _cd.head.in_features == N_PATCHES * D_MODEL, f"CD head wrong: {_cd.head}"
assert _cd.head.out_features == PRED_LEN
if "CD_Head" in MODES:
    _h = build_model("CD_Head")
    assert _h.head.in_features == N_PATCHES * D_MODEL + D_MODEL, f"CD_Head head wrong: {_h.head}"
    enc_cd   = sum(p.numel() for p in _cd.encoder.parameters())
    enc_head = sum(p.numel() for p in _h.encoder.parameters())
    assert enc_cd == enc_head, "CD_Head encoder must match CD encoder exactly"
    print(f"CD_Head OK: head {_h.head}, encoder params identical to CD ({enc_cd})")
    del _h
if "DLinear" in MODES:
    _dl = build_model("DLinear")
    assert _dl.linear_trend is not _dl.linear_remainder, "DLinear branches must be independent"
    del _dl
print(f"Architecture assertions passed. N_PATCHES={N_PATCHES}  CD head={_cd.head}")
del _cd, _x
free_cuda()


In [ ]:
# Training engine: epoch-based early stopping (identical schedule to the canonical grid).

def _cosine_warmup(optimizer, epoch: int, warmup: int) -> None:
    if epoch < warmup:
        lr = LR * (epoch + 1) / warmup
    else:
        progress = (epoch - warmup) / max(1, MAX_EPOCHS - warmup)
        lr = LR * 0.5 * (1.0 + np.cos(np.pi * progress))
    for g in optimizer.param_groups:
        g["lr"] = lr


@torch.inference_mode()
def _evaluate(model: nn.Module, loader: DataLoader) -> tuple[float, float]:
    model.eval()
    mse = mae = n = 0.0
    for xb, yb in loader:
        pred = model(xb.to(DEVICE)).cpu()
        mse += nn.functional.mse_loss(pred, yb, reduction="sum").item()
        mae += nn.functional.l1_loss(pred, yb, reduction="sum").item()
        n   += yb.numel()
    return mse / n, mae / n


def _fit(mode: str, seed: int, datasets: tuple, batch_size: int) -> dict:
    x_tr, y_tr, x_va, y_va, x_te, y_te = datasets
    train_dl = DataLoader(TensorDataset(x_tr, y_tr), batch_size=batch_size, shuffle=True,  drop_last=False)
    val_dl   = DataLoader(TensorDataset(x_va, y_va), batch_size=batch_size, shuffle=False, drop_last=False)
    test_dl  = DataLoader(TensorDataset(x_te, y_te), batch_size=batch_size, shuffle=False, drop_last=False)

    use_warmup = mode in ("CI", "CD", "CD_Head")
    model     = build_model(mode).to(DEVICE)
    opt       = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler    = GradScaler("cuda", enabled=(DEVICE.type == "cuda"))
    criterion = nn.MSELoss()

    spe = len(train_dl)
    best_val, best_epoch, best_steps = float("inf"), 0, 0
    best_state, no_improve, total = None, 0, 0
    try:
        for epoch in range(MAX_EPOCHS):
            if use_warmup:
                _cosine_warmup(opt, epoch, WARMUP_EPOCHS)
            model.train()
            for xb, yb in train_dl:
                opt.zero_grad(set_to_none=True)
                with autocast("cuda", enabled=(DEVICE.type == "cuda")):
                    loss = criterion(model(xb.to(DEVICE)), yb.to(DEVICE))
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(opt)
                scaler.update()
                total += 1
            val_mse, _ = _evaluate(model, val_dl)
            if val_mse < best_val:
                best_val, best_epoch, best_steps = val_mse, epoch + 1, total
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= PATIENCE:
                    break
        if best_state:
            model.load_state_dict(best_state)
        test_mse, test_mae = _evaluate(model, test_dl)
    finally:
        del model, opt, scaler
        free_cuda()
    return {"test_mse": test_mse, "test_mae": test_mae, "best_epoch": best_epoch,
            "batch_size": batch_size, "steps_per_epoch": spe, "total_steps": best_steps}


In [ ]:
# Main sweep with atomic checkpointing and safe resume.
OUT = Path("/kaggle/working/results_block_cov.csv")
TMP = OUT.with_suffix(".csv.tmp")


def _key(cell: str, mode: str, seed: int) -> tuple:
    return (str(cell), str(mode), int(seed))


def _save(rows: list[dict]) -> None:
    pd.DataFrame(rows).to_csv(TMP, index=False)
    os.replace(TMP, OUT)


def train_one(cell_meta: dict, mode: str, seed: int) -> dict:
    set_seed(seed)
    raw = generate_block(cell_meta["C"], cell_meta["rho_in"], RHO_OUT, GROUP_SIZE, seed)
    tr, va, te = split_normalise(raw)
    datasets = (*make_windows(tr), *make_windows(va), *make_windows(te))
    batch = BATCH_BY_C[cell_meta["C"]][mode]
    while True:
        try:
            set_seed(seed)
            metrics = _fit(mode, seed, datasets, batch)
            break
        except RuntimeError as exc:
            if "out of memory" not in str(exc).lower():
                raise
            free_cuda()
            if batch == 1:
                raise
            batch = max(1, batch // 2)
            print(f"  OOM: retrying at batch_size={batch}")
    return {"dataset": "block_cov_ar1", "cell": cell_meta["cell"], "C": cell_meta["C"],
            "rho_in": cell_meta["rho_in"], "rho_out": RHO_OUT, "group_size": GROUP_SIZE,
            "mode": mode, "seed": seed, **metrics}


if OUT.exists() and OUT.stat().st_size > 100:
    _existing = pd.read_csv(OUT)
    done    = {_key(r.cell, r.mode, r.seed) for r in _existing.itertuples()}
    results = _existing.to_dict("records")
    print(f"Resuming: {len(done)} runs done.")
else:
    done, results = set(), []

total = len(CELLS) * len(MODES) * len(SEEDS)
idx = 0
fails = []
for cell_meta in CELLS:
    for mode in MODES:
        for seed in SEEDS:
            idx += 1
            key = _key(cell_meta["cell"], mode, seed)
            if key in done:
                print(f"[{idx}/{total}] SKIP {cell_meta['cell']} {mode} seed={seed}")
                continue
            print(f"[{idx}/{total}] {cell_meta['cell']} {mode} seed={seed} ...", end=" ", flush=True)
            t0 = time.time()
            try:
                row = train_one(cell_meta, mode, seed)
            except Exception as exc:
                free_cuda()
                print(f"FAILED: {type(exc).__name__}: {exc}")
                fails.append((cell_meta["cell"], mode, seed, repr(exc)))
                continue
            print(f"mse={row['test_mse']:.4f}  epoch={row['best_epoch']}  bs={row['batch_size']}  ({time.time()-t0:.0f}s)")
            results.append(row)
            done.add(key)
            _save(results)

print(f"\nDone. {len(results)}/{total} -> {OUT}")
for f in fails:
    print("  FAIL", f)


In [ ]:
# Summary: per-cell mode means and CD/CI, DLinear/CI ratios.
df = pd.read_csv(OUT)
print(f"Rows: {len(df)}  modes: {sorted(df['mode'].unique())}  cells: {sorted(df['cell'].unique())}")
piv = df.groupby(["C", "rho_in", "mode"])["test_mse"].mean().unstack("mode")
out = piv.copy()
out["CD/CI"] = (piv["CD"] / piv["CI"]).round(4)
if "DLinear" in piv.columns:
    out["DLinear/CI"] = (piv["DLinear"] / piv["CI"]).round(4)
print("\nMean test MSE by cell (CD/CI > 1.0 favours CI):")
print(out.round(4).to_string())


In [ ]:
from IPython.display import FileLink
FileLink(str(OUT))
